# 02 Datasets and Reproducibility

Create tiny synthetic pair and retrieval datasets, adapt tabular inputs, evaluate metrics, run resampling, and write reproducibility metadata.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

MATHEEL_REF = "v0.4.2"


def run(*args, cwd=None):
    subprocess.check_call(list(args), cwd=cwd)

run(sys.executable, "-m", "pip", "install", "jedi>=0.16,<1")
run("rm", "-rf", "/content/matheel")
run("git", "clone", "https://github.com/FahadEbrahim/matheel.git", "/content/matheel")
run("git", "checkout", MATHEEL_REF, cwd="/content/matheel")
run(sys.executable, "-m", "pip", "install", "-e", ".[all]", cwd="/content/matheel")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matheel_mplconfig")
print(f"Matheel installed from {MATHEEL_REF}")


In [ ]:
%cd /content/matheel
import json
from pathlib import Path

import pandas as pd

from matheel.datasets import load_pair_datasets, load_retrieval_datasets, write_pair_dataset
from matheel.evaluation import evaluate_pair_dataset, evaluate_pair_resamples, evaluate_retrieval_dataset
from matheel.resampling import kfold_splits

workspace = Path("/content/matheel_dataset_demo")
workspace.mkdir(parents=True, exist_ok=True)

In [ ]:
pair_root = workspace / "normalized_pairs"
write_pair_dataset(
    pair_root,
    files=pd.DataFrame([
        {"file_id": "a", "text": "print(1)", "suffix": ".py"},
        {"file_id": "b", "text": "print(1)", "suffix": ".py"},
        {"file_id": "c", "text": "print(2)", "suffix": ".py"},
    ]),
    pairs=pd.DataFrame([
        {"left_id": "a", "right_id": "b", "label": 1},
        {"left_id": "a", "right_id": "c", "label": 0},
    ]),
)
scored_pairs, pair_metrics = evaluate_pair_dataset(pair_root, threshold=0.5, similarity_options={"feature_weights": {"levenshtein": 1.0}})
display(scored_pairs)
print(json.dumps(pair_metrics, indent=2, sort_keys=True))

In [ ]:
raw_pair_root = workspace / "raw_pairs"
raw_pair_root.mkdir(exist_ok=True)
pd.DataFrame([
    {"left_code": "return x + 1", "right_code": "return x + 1", "label": 1},
    {"left_code": "return x + 1", "right_code": "return y - 1", "label": 0},
]).to_csv(raw_pair_root / "pairs.csv", index=False)

adapted_pairs = load_pair_datasets([
    {
        "source": "local",
        "identifier": raw_pair_root,
        "name": "tiny_tabular_pairs",
        "adapter": "auto_pair_tabular",
        "adapter_options": {
            "pair_table": "pairs.csv",
            "left_text_column": "left_code",
            "right_text_column": "right_code",
            "label_column": "label",
        },
    }
])
adapted_scored, adapted_metrics = evaluate_pair_dataset(adapted_pairs, threshold=0.5, similarity_options={"feature_weights": {"levenshtein": 1.0}})
display(adapted_scored)
print(json.dumps(adapted_metrics, indent=2, sort_keys=True))

In [ ]:
raw_retrieval_root = workspace / "raw_retrieval"
raw_retrieval_root.mkdir(exist_ok=True)
pd.DataFrame([
    {"query_id": "q1", "document_id": "d1", "query_code": "print(1)", "document_code": "print(1)", "relevance": 1},
    {"query_id": "q1", "document_id": "d2", "query_code": "print(1)", "document_code": "print(2)", "relevance": 0},
    {"query_id": "q2", "document_id": "d2", "query_code": "print(2)", "document_code": "print(2)", "relevance": 1},
]).to_csv(raw_retrieval_root / "retrieval.csv", index=False)

retrieval_dataset = load_retrieval_datasets([
    {
        "source": "local",
        "identifier": raw_retrieval_root,
        "name": "tiny_tabular_retrieval",
        "adapter": "auto_retrieval_tabular",
        "adapter_options": {
            "retrieval_table": "retrieval.csv",
            "query_text_column": "query_code",
            "document_text_column": "document_code",
            "relevance_column": "relevance",
        },
    }
])
retrieval_scored, retrieval_metrics = evaluate_retrieval_dataset(retrieval_dataset, k=1, similarity_options={"feature_weights": {"levenshtein": 1.0}})
display(retrieval_scored.sort_values(["query_id", "document_id"]))
print(json.dumps(retrieval_metrics, indent=2, sort_keys=True))

In [ ]:
splits = kfold_splits(len(scored_pairs), n_splits=2, seed=7, labels=scored_pairs["label"].tolist())
fold_metrics, fold_summary = evaluate_pair_resamples(scored_pairs, splits, threshold=0.5)
display(fold_metrics)
display(fold_summary)

In [ ]:
from examples.evaluation.reproducible_benchmark_demo import run_benchmark

artifacts = run_benchmark(workspace / "reproducible_benchmark")
for name, path in artifacts.items():
    print(name, path)